In [1]:
pip install transformers datasets torch scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.4/491.4 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 53.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 39.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 34.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 71.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
import torch
from transformers import T5Tokenizer, T5ForConditionalGeneration, Trainer, TrainingArguments
from datasets import Dataset

In [3]:
# Step 1: Prepare the data

data = [
    # Apple - Fresh
    {"input_text": "fruit: apple | status: fresh", "target_summary": "The apple is fresh, with a crisp texture and bright red color. It has a firm bite and a slightly sweet, tangy flavor. Ideal for eating raw, adding to salads, or baking. Its skin is smooth and unblemished."},
    {"input_text": "fruit: apple | status: fresh", "target_summary": "This apple appears very fresh. Its vibrant color and firm flesh indicate peak ripeness. It is juicy and aromatic, great for snacks or desserts. There's no sign of bruising or softness."},
    {"input_text": "fruit: apple | status: fresh", "target_summary": "A fresh apple typically has a crunchy texture and glossy skin. This one is no different—it is smooth, brightly colored, and smells fruity. It tastes slightly tart and refreshing. Perfect for daily consumption or fruit bowls."},
    {"input_text": "fruit: apple | status: fresh", "target_summary": "The apple looks recently harvested and in excellent condition. There's a firm snap when bitten, and the inside is juicy and creamy white. Great for raw use or pairing with peanut butter. No signs of decay or wrinkles are visible."},
    {"input_text": "fruit: apple | status: fresh", "target_summary": "This fresh apple has a pleasant scent and feels heavy in the hand. It is crunchy and sweet, with a clean outer skin. Ideal for eating, baking, or making cider. No soft spots or blemishes are seen."},

    # Apple - Stale
    {"input_text": "fruit: apple | status: stale", "target_summary": "The apple seems stale and past its prime. Its skin may be wrinkled or discolored, and the flesh could feel soft or mushy. There's a chance of internal browning or a fermented smell. Not suitable for fresh consumption."},
    {"input_text": "fruit: apple | status: stale", "target_summary": "This apple is likely stale and deteriorating. It might have dark bruises, wrinkled skin, or feel unusually soft. The taste could be bland or sour. It's best discarded or used in compost."},
    {"input_text": "fruit: apple | status: stale", "target_summary": "A stale apple loses its crispness and firmness. The color often fades, and pressure easily causes dents. It may have a fermented odor and brownish spots inside. Avoid using it in raw recipes."},
    {"input_text": "fruit: apple | status: stale", "target_summary": "This apple is stale and possibly spoiled. It may have shriveled skin and internal soft patches. Any fruity aroma is likely gone, replaced by an off-putting smell. It should not be eaten fresh."},
    {"input_text": "fruit: apple | status: stale", "target_summary": "Stale apples often turn soft and mealy. Their skin may become dull and feel sticky. The core may brown, and the flavor turns acidic or tasteless. Such fruit is unsuitable for eating raw."},

    # Banana - Fresh
    {"input_text": "fruit: banana | status: fresh", "target_summary": "This banana is fresh and ripe, with a vibrant yellow peel and slightly firm touch. It has a sweet aroma and smooth texture. Perfect for snacking or adding to cereal. No bruising or discoloration is present."},
    {"input_text": "fruit: banana | status: fresh", "target_summary": "A fresh banana is easy to spot—it’s evenly yellow and gives slightly under pressure. It has a mellow sweetness and soft but intact flesh. Great for smoothies or eating directly. The peel has no spots or splits."},
    {"input_text": "fruit: banana | status: fresh", "target_summary": "This banana looks ideal for consumption. The color is bright, and it feels slightly soft to the touch. Its natural sugars are at the right balance. The peel is smooth and intact with no blemishes."},
    {"input_text": "fruit: banana | status: fresh", "target_summary": "The banana is perfectly fresh, with no signs of ripening beyond ideal. It is easy to peel, and the inside is creamy and sweet. There's a pleasant tropical scent. Excellent for quick energy and healthy snacks."},
    {"input_text": "fruit: banana | status: fresh", "target_summary": "Fresh bananas have vibrant peels and a light fruity smell. This one is mildly soft but not mushy, making it perfect for direct consumption. It tastes sweet and creamy. It’s clean and free from dark spots."},

    # Banana - Stale
    {"input_text": "fruit: banana | status: stale", "target_summary": "This banana is stale and likely overripe. It may have multiple dark spots and a sagging peel. The flesh is overly soft and may leak sugary fluids. It’s suitable for baking but not fresh use."},
    {"input_text": "fruit: banana | status: stale", "target_summary": "A stale banana has lost its firmness and may have split open. The smell could be sour or fermented. Dark bruising is common, and it may attract fruit flies. Best used in compost or baking recipes."},
    {"input_text": "fruit: banana | status: stale", "target_summary": "The banana looks stale and is past its edible phase. The peel is likely blackened or spotted, and the inside is overly soft. Texture and flavor both suffer. Not safe for direct consumption."},
    {"input_text": "fruit: banana | status: stale", "target_summary": "This banana is no longer fresh. It is too mushy to eat raw, and the peel is discolored. A strange smell may be present. It could be salvaged for banana bread, but fresh eating is discouraged."},
    {"input_text": "fruit: banana | status: stale", "target_summary": "Stale bananas often turn brown and emit a strong odor. The peel might feel sticky or cracked. Its taste is overly sweet and the texture slimy. Best used for baking or thrown away."},

    # Orange - Fresh
    {"input_text": "fruit: orange | status: fresh", "target_summary": "This orange is fresh, heavy for its size, and has a glossy, firm peel. It smells citrusy and feels resilient when pressed. The flesh inside is likely juicy and sweet. Great for juicing or eating raw."},
    {"input_text": "fruit: orange | status: fresh", "target_summary": "A fresh orange has a vibrant orange color and smooth texture. It feels firm and has no visible dents. The aroma is zesty and inviting. Ideal for juice, desserts, or snacking."},
    {"input_text": "fruit: orange | status: fresh", "target_summary": "This orange is in excellent condition. It has a firm outer layer and a fragrant scent. The inside is juicy, sweet, and slightly tangy. Perfect for a refreshing vitamin C boost."},
    {"input_text": "fruit: orange | status: fresh", "target_summary": "The orange is fresh and full of juice. Its peel is intact with no mold or soft spots. The weight feels right for its size, a sign of internal juiciness. Use it for fresh juice or salads."},
    {"input_text": "fruit: orange | status: fresh", "target_summary": "Fresh oranges have firm skin and a bright appearance. This one smells citrusy and feels dense. The interior is likely bursting with juice and flavor. It's perfect for hydration and nutrition."},

    # Orange - Stale
    {"input_text": "fruit: orange | status: stale", "target_summary": "This orange appears stale and degraded. The skin may show mold or soft areas. It feels light and may be dry inside. Discard if it smells fermented or appears squishy."},
    {"input_text": "fruit: orange | status: stale", "target_summary": "Stale oranges often feel soft or hollow and may show wrinkles. The color might look faded, and the scent unpleasant. The juice content is likely reduced. It's best to throw it out."},
    {"input_text": "fruit: orange | status: stale", "target_summary": "This orange is stale and past consumption quality. The skin is likely dull and thin in places. Internally, it may be dry or moldy. Not recommended for juicing or eating."},
    {"input_text": "fruit: orange | status: stale", "target_summary": "A stale orange shows signs of spoilage like dark patches or softness. The aroma may be sour instead of citrusy. Inside, the segments might have shriveled. Avoid consuming it raw."},
    {"input_text": "fruit: orange | status: stale", "target_summary": "The orange is stale and should not be consumed fresh. It feels unusually soft and possibly has mold on the surface. Any juice left inside may taste bitter. Discard if there's any doubt."},
]


In [4]:
prefix = "summarize: "

# Step 2: Tokenization

tokenizer = T5Tokenizer.from_pretrained("t5-small")

def preprocess(example):
    input_text = prefix + example["input_text"]
    target_text = example["target_summary"]
    model_input = tokenizer(input_text, max_length=64, truncation=True, padding="max_length")
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(target_text, max_length=64, truncation=True, padding="max_length")
    model_input["labels"] = labels["input_ids"]
    return model_input

dataset = Dataset.from_list(data)
tokenized_dataset = dataset.map(preprocess)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


Map:   0%|          | 0/30 [00:00<?, ? examples/s]

/usr/local/lib/python3.11/dist-packages/transformers/tokenization_utils_base.py:3980: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(


In [5]:
pip install --upgrade transformers

In [6]:
model = T5ForConditionalGeneration.from_pretrained("t5-small")

training_args = TrainingArguments(
    output_dir="./t5_fruit_summary",
    per_device_train_batch_size=2,
    num_train_epochs=20,
    learning_rate=3e-4,
    weight_decay=0.01,
    logging_steps=5,
    save_strategy="epoch",
    save_total_limit=2,
)

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [7]:
pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 4.4 MB/s eta 0:00:00


In [8]:
pip install rouge_score

  Preparing metadata (setup.py) ... done
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=c7cfb43c0573d290cc30ec2adafbd28452822a38a63b76b7aa579773e456c44c
  Stored in directory: /root/.cache/pip/wheels/1e/19/43/8a442dc83660ca25e163e1bd1f89919284ab0d0c1475475148
Successfully built rouge_score


In [9]:
import evaluate

rouge = evaluate.load("rouge")

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    result = rouge.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)
    return {key: value.mid.fmeasure * 100 for key, value in result.items()}


In [10]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    compute_metrics=compute_metrics

)

In [11]:
trainer.train()

wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: anjalitirumala04 (anjalitirumala04-kmce) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


Step,Training Loss
5,5.672600
10,3.390500
15,3.342100
20,2.854200
25,2.638600
30,2.547000
35,2.382800
40,2.368100
45,2.208600
50,1.928600


TrainOutput(global_step=300, training_loss=1.6328028186162313, metrics={'train_runtime': 172.9098, 'train_samples_per_second': 3.47, 'train_steps_per_second': 1.735, 'total_flos': 10150635110400.0, 'train_loss': 1.6328028186162313, 'epoch': 20.0})

In [12]:
def generate_summary(input_text):
    input_ids = tokenizer("summarize: " + input_text, return_tensors="pt").input_ids
    # Move input_ids to the same device as the model
    input_ids = input_ids.to(model.device)
    output_ids = model.generate(input_ids, max_length=64, num_beams=4, early_stopping=True)
    return tokenizer.decode(output_ids[0], skip_special_tokens=True)

In [15]:
# Example usage
print(generate_summary("fruit: apple | status: stale"))
print(generate_summary("fruit: banana | status: fresh"))

A stale apple is likely stale or deteriorated. It may have a deterioration or smelly. It may have a shriveled smell. It's best to eat raw raw.
The banana looks fresh and has a vibrant aroma. It feels soft and creamy, with no signs of ripening. It has a sweet texture and smooth texture. Perfect for juicing or eating.
